# 17. Iso-catalog phase allocation and paired inference

![Iso-catalog allocation and paired inference](../images/17_hierarchical_support_and_factorial_inference.svg)

This capstone runs a miniature version of the study in code: three allocations that share one nominal catalog, a matched nearby-jitter diagnostic, and a paired contrast computed over eight trained-model blocks. The numbers are synthetic, but every shape and every reduction matches the real analysis.

Work through the [lecture](../lectures/17_hierarchical_support_and_factorial_inference.md) first, then return to the [tutorial index](../README.md).

**Learning goals:** tell exposure apart from nominal catalog size, keep the eight paired allocation blocks intact through every reduction, calculate the GFC-minus-completion residual, read the phase-versus-jitter diagnostic for what it does and does not rule out, and resist counting participants as extra trained models.

In [ ]:
import numpy as np
from scipy import stats

SEED = 41
rng = np.random.default_rng(SEED)
assert rng.random() >= 0.0


## 1. Same nominal catalog, different allocation

`U` is the number of eligible sequences a model may draw from and `k` is the number of phase origins chosen inside each of them, so `U × k` counts the sequence-origin atoms on offer. The cell below checks that all three allocations reach the same 250,000 atoms and therefore the same planned recurrence under one fixed clip exposure.

Equal counts are a control, not a claim. Two atoms from the same sequence can overlap heavily while two atoms from different sequences cannot, which is why the real study also runs phase and duplicate audits.

In [ ]:
ALLOCATIONS = ('breadth', 'balanced', 'phase_depth')
U = np.array([250_000, 125_000, 62_500])
K = np.array([1, 2, 4])
assert np.all(U * K == 250_000)
EXPOSURE = 4_096_000
recurrence = EXPOSURE / (U * K)
assert np.allclose(recurrence, recurrence[0])
print(dict(zip(ALLOCATIONS, recurrence)))


## 2. Simulate eight paired model blocks

The array below is built the way the real scores are organized: one axis for the eight training blocks, one for the three allocations, and one for participants. A block effect is shared by all three allocations in that block, which is exactly the pairing that a bootstrap must preserve later.

Read the shape as its meaning. `(8, 3, 32)` is eight trained-model blocks, three trained models each, and 32 repeated participant measurements per model. It is not 768 independent observations.

In [ ]:
blocks, participants, cells = 8, 32, 3
block_effect = rng.normal(0, 0.03, size=(blocks, 1, 1))
participant_effect = rng.normal(0, 0.05, size=(1, 1, participants))
allocation_effect = np.array([0.00, 0.01, 0.04])[None, :, None]
GFC = 0.18 + block_effect + participant_effect + allocation_effect + rng.normal(0, 0.02, size=(blocks, cells, participants))
completion = 0.14 + block_effect + participant_effect + np.array([0.00, 0.01, 0.015])[None, :, None] + rng.normal(0, 0.02, size=(blocks, cells, participants))
assert GFC.shape == completion.shape == (8, 3, 32)


## 3. The primary residual contrast

Two steps, in this order. First, inside every block and allocation, subtract the independent-completion margin from the GFC margin to get a residual `D`. Second, inside every block, subtract the breadth residual from the phase-depth residual to get one number `P`.

Doing it in that order matters. The subtraction of the control happens on a shared scale, and the comparison of allocations happens inside the block that produced both models. The result is eight paired values and a Student `t` interval with seven degrees of freedom, no matter how many participants were evaluated.

In [ ]:
G = GFC.mean(axis=2)
C = completion.mean(axis=2)
D = G - C
P = D[:, 2] - D[:, 0]
mean_P = P.mean()
se_P = P.std(ddof=1) / np.sqrt(blocks)
critical = stats.t.ppf(0.975, df=blocks - 1)
interval = (mean_P - critical * se_P, mean_P + critical * se_P)
print('primary residual contrast', mean_P, interval)
assert P.shape == (8,)


## 4. Semantic phase depth versus nearby jitter

Jitter keeps the same sequences, the same four origins per sequence, and the same exposure, and moves those four origins next to each other instead of spreading them around the gait cycle. If phase depth beats it, the advantage cannot be explained by simply drawing four different start indices.

Only four prespecified blocks train the jitter model, so the diagnostic is deliberately less precise than the primary contrast. Report its uncertainty and never treat it as a fourth point on the allocation path.

In [ ]:
jitter = GFC[:4, 2] - 0.025 + rng.normal(0, 0.01, size=(4, participants))
phase_minus_jitter = GFC[:4, 2].mean(axis=1) - jitter.mean(axis=1)
assert phase_minus_jitter.shape == (4,)
assert np.isfinite(phase_minus_jitter).all()
print('paired phase minus jitter', phase_minus_jitter)


## 5. A sensitivity bootstrap preserves the hierarchy

The bootstrap below draws whole blocks, so a selected block arrives with its complete breadth, balanced, and phase-depth triplet still attached. It also draws one set of participant indices and applies that same set to every cell, which keeps each participant's full profile intact.

Notice what it never does: it never resamples individual model rows. That would let a breadth model from one block sit beside a phase-depth model from another, which throws away the covariance that pairing created. The bootstrap is a sensitivity check on the eight observed blocks, not a way to manufacture new training runs.

In [ ]:
def crossed_bootstrap(gfc, independent, draws=500, seed=43):
    local = np.random.default_rng(seed)
    values = []
    for _ in range(draws):
        block_ids = local.integers(0, gfc.shape[0], size=gfc.shape[0])
        participant_ids = local.integers(0, gfc.shape[2], size=gfc.shape[2])
        residual = (gfc[block_ids][:, :, participant_ids].mean(axis=2) - independent[block_ids][:, :, participant_ids].mean(axis=2))
        values.append((residual[:, 2] - residual[:, 0]).mean())
    return np.asarray(values)

boot = crossed_bootstrap(GFC, completion)
assert boot.shape == (500,) and np.isfinite(boot).all()


## 6. Interpretation

A phase-depth advantage on its own is a result about this corpus, this objective, and this instrument. It is not a general law about video data.

The strongest available statement needs four things to agree: phase separation must beat matched jitter, the continuous margin must point the same way as top-1 and mean reciprocal rank, the effect must survive subtracting independent completion, and the locked geometry diagnostic must not contradict it. The check below tests the directional part of that agreement.

In [ ]:
top1_scores = np.clip(
    0.54 + 0.65 * allocation_effect + 0.25 * block_effect + rng.normal(0, 0.015, size=(blocks, cells, participants)),
    0,
    1,
)
mrr_scores = np.clip(
    0.66 + 0.45 * allocation_effect + 0.20 * block_effect + rng.normal(0, 0.012, size=(blocks, cells, participants)),
    0,
    1,
)
top1_direction = np.sign(top1_scores[:, 2].mean() - top1_scores[:, 0].mean())
mrr_direction = np.sign(mrr_scores[:, 2].mean() - mrr_scores[:, 0].mean())
margin_direction = np.sign((G[:, 2] - G[:, 0]).mean())
concordant = top1_direction == mrr_direction == margin_direction
assert concordant
print('directions:', {'top1': top1_direction, 'mrr': mrr_direction, 'margin': margin_direction})
print('directional concordance:', bool(concordant))


**Takeaway:** the study changes where a fixed nominal catalog lives in the video hierarchy, holding the count of atoms and the clip exposure constant. Its inference unit is the paired block, so eight numbers carry the headline no matter how large the score tensor grows. Its claim is bounded by three things: the semantic phase audit, the matched jitter diagnostic, and the comparability of the GFC and control measurements.

Previous: [16. Reproducible evaluators](16_reproducible_scientific_evaluators.ipynb)